In [1]:
import sys
import subprocess
import pyspark
print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

Python: 3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]
PySpark: 4.2.0
openjdk version "17.0.12" 2024-07-16
OpenJDK Runtime Environment Temurin-17.0.12+7 (build 17.0.12+7)
OpenJDK 64-Bit Server VM Temurin-17.0.12+7 (build 17.0.12+7, mixed mode, sharing)



In [ ]:
from pathlib import Path
import pandas as pd


BASE    = Path(r"D:\Documents\Learning\7\ТИАБД\задания\3")
ARCHIVE = BASE / "Транзакции КОМУС" / "archive"
TX_DIR  = BASE / "transactions"

STAGING = BASE / "spark_input"
STAGING.mkdir(exist_ok=True)

FILES = {
    "clients":     ARCHIVE / "Клиенты.csv",
    "sales":       ARCHIVE / "Продажи.csv",
    "products":    ARCHIVE / "Продукты.csv",
    "receipts":    ARCHIVE / "Суммы чеков.csv",
    "transaction": TX_DIR / "transactions.csv",
}

for name, path in FILES.items():
    if not path.exists():
        print(f"[SKIP] {name}: не найден -> {path}")
        continue

    df = None
    for sep in ["\t", ";", ","]:
        for enc in ["utf-8", "cp1251"]:
            try:
                df_try = pd.read_csv(path, sep=sep, encoding=enc, dtype=str,
                                     on_bad_lines="skip", engine="python")
                if df_try.shape[1] > 1:
                    df = df_try
                    print(f"{name}: sep={repr(sep)}, enc={enc}, cols={df.shape[1]}")
                    break
            except Exception:
                continue
        if df is not None:
            break

    if df is None:
        print(f"[FAIL] {name}: не удалось распарсить {path}")
        continue

    out = STAGING / f"{name}.csv"
    df.to_csv(out, index=False, encoding="utf-8")
    print(f"  -> {out.name}  rows={len(df)}  columns={list(df.columns)}")

clients: sep='\t', enc=utf-8, cols=3
  -> clients.csv  rows=3838  columns=['Client_ID', 'Покупатель', 'Contractor_name']
sales: sep='\t', enc=utf-8, cols=11
  -> sales.csv  rows=865222  columns=['Себестоимость', 'Дата покупки', 'Номер чека', 'Product_key', 'Филиал', 'Количество', 'Сумма покупки', 'Сумма скидки', 'Client_ID', 'Машина доставки', 'Адрес']
products: sep='\t', enc=utf-8, cols=4
  -> products.csv  rows=21163  columns=['Product_Key', 'Название товара', 'Название бренда', 'Группа товара']
receipts: sep='\t', enc=utf-8, cols=7
  -> receipts.csv  rows=160114  columns=['Дата покупки', 'Client_ID', 'Адрес', 'Машина доставки', 'Филиал', 'Номер чека', 'Сумма покупки|Сумма']
transaction: sep='\t', enc=utf-8, cols=11
  -> transaction.csv  rows=3276748  columns=['Дата транзакции', 'Магазин', 'Чек', 'Клиент', 'Товар', 'Группа', 'Подгруппа', 'Кол-во', 'Сумма без скидки', 'Сумма продажи', 'Маржа']


In [32]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (SparkSession.builder
    .appName("Practice3_Transactions")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Master:", spark.sparkContext.master)
print("Spark:", spark.version)

Master: local[*]
Spark: 4.2.0


In [8]:
tx = (spark.read
    .option("header", True)
    .option("sep", ",")
    .option("inferSchema", True)
    .csv(str(STAGING / "transaction.csv")))

tx.printSchema()
print("Строк:", tx.count())
tx.show(5, truncate=False)

root
 |-- Дата транзакции: string (nullable = true)
 |-- Магазин: string (nullable = true)
 |-- Чек: string (nullable = true)
 |-- Клиент: string (nullable = true)
 |-- Товар: string (nullable = true)
 |-- Группа: string (nullable = true)
 |-- Подгруппа: string (nullable = true)
 |-- Кол-во: double (nullable = true)
 |-- Сумма без скидки: integer (nullable = true)
 |-- Сумма продажи: integer (nullable = true)
 |-- Маржа: double (nullable = true)

Строк: 3276748
+---------------+-------+------------+--------+---------+--------+---------+------+----------------+-------------+-----+
|Дата транзакции|Магазин|Чек         |Клиент  |Товар    |Группа  |Подгруппа|Кол-во|Сумма без скидки|Сумма продажи|Маржа|
+---------------+-------+------------+--------+---------+--------+---------+------+----------------+-------------+-----+
|01/04/2020     |Store01|code00350356|cl100636|sku006110|group024|subgr0456|1.0   |57              |57           |17.5 |
|01/04/2020     |Store01|code00350356|cl100636|sku

In [ ]:
tx_clean = tx
rename_map = {
    "Дата транзакции":     "tx_date",
    "Магазин":             "store",
    "Чек":                 "receipt",
    "Клиент":              "client",
    "Товар":               "product",
    "Группа":              "group_name",
    "Подгруппа":           "subgroup",
    "Кол-во":              "qty",
    "Сумма без скидки":    "amount_no_discount",
    "Сумма продажи":       "sale_amount",
    "Маржа":               "margin",
}
for old, new in rename_map.items():
    if old in tx_clean.columns:
        tx_clean = tx_clean.withColumnRenamed(old, new)


tx_clean = (tx_clean
    .withColumn("tx_date",            F.to_date("tx_date", "dd/MM/yyyy"))
    .withColumn("qty",                F.col("qty").cast("int"))
    .withColumn("amount_no_discount", F.col("amount_no_discount").cast("double"))
    .withColumn("sale_amount",        F.col("sale_amount").cast("double"))
    .withColumn("margin",             F.col("margin").cast("double"))
)


tx_clean = tx_clean.withColumn(
    "margin_pct",
    F.when(F.col("sale_amount") > 0,
           F.round(F.col("margin") / F.col("sale_amount") * 100, 2))
)

tx_clean.printSchema()
tx_clean.show(5, truncate=False)

root
 |-- tx_date: date (nullable = true)
 |-- store: string (nullable = true)
 |-- receipt: string (nullable = true)
 |-- client: string (nullable = true)
 |-- product: string (nullable = true)
 |-- group_name: string (nullable = true)
 |-- subgroup: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- amount_no_discount: double (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- margin: double (nullable = true)
 |-- margin_pct: double (nullable = true)

+----------+-------+------------+--------+---------+----------+---------+---+------------------+-----------+------+----------+
|tx_date   |store  |receipt     |client  |product  |group_name|subgroup |qty|amount_no_discount|sale_amount|margin|margin_pct|
+----------+-------+------------+--------+---------+----------+---------+---+------------------+-----------+------+----------+
|2020-04-01|Store01|code00350356|cl100636|sku006110|group024  |subgr0456|1  |57.0              |57.0       |17.5  |30.7      |
|2020

In [16]:
sales_schema = StructType([
    StructField("cost",          StringType(), True),
    StructField("purchase_date", StringType(), True),
    StructField("receipt_no",    StringType(), True),
    StructField("product_key",   StringType(), True),
    StructField("branch",        StringType(), True),
    StructField("quantity",      StringType(), True),
    StructField("sum_purchase",  StringType(), True),
    StructField("discount",      StringType(), True),
    StructField("client_id",     StringType(), True),
    StructField("delivery_car",  StringType(), True),
    StructField("address",       StringType(), True),
])

sales = (spark.read
    .option("header", True)
    .option("sep", ",")
    .schema(sales_schema)
    .csv(str(STAGING / "sales.csv")))

sales = (sales
    .withColumn("cost",          F.regexp_replace("cost", ",", ".").cast("double"))
    .withColumn("quantity",      F.col("quantity").cast("int"))
    .withColumn("sum_purchase",  F.regexp_replace("sum_purchase", ",", ".").cast("double"))
    .withColumn("discount",      F.regexp_replace("discount", ",", ".").cast("double"))
    .withColumn("purchase_date", F.to_date("purchase_date", "dd.MM.yyyy")))

sales.printSchema()
print("Строк:", sales.count())
sales.show(5, truncate=False)

root
 |-- cost: double (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- receipt_no: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sum_purchase: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- client_id: string (nullable = true)
 |-- delivery_car: string (nullable = true)
 |-- address: string (nullable = true)

Строк: 865222
+------+-------------+----------+-----------+-----------------------------+--------+------------+--------+---------+------------+---------------------------------------------+
|cost  |purchase_date|receipt_no|product_key|branch                       |quantity|sum_purchase|discount|client_id|delivery_car|address                                      |
+------+-------------+----------+-----------+-----------------------------+--------+------------+--------+---------+------------+---------------------------------------------+
|

In [17]:
clients = (spark.read
    .option("header", True)
    .option("sep", ",")
    .option("encoding", "UTF-8")
    .csv(str(STAGING / "clients.csv")))

products = (spark.read
    .option("header", True)
    .option("sep", ",")
    .option("encoding", "UTF-8")
    .csv(str(STAGING / "products.csv")))

receipts = (spark.read
    .option("header", True)
    .option("sep", ",")
    .option("encoding", "UTF-8")
    .csv(str(STAGING / "receipts.csv")))

for name, df in [("clients", clients), ("products", products), ("receipts", receipts)]:
    print("===", name, "===")
    print("Колонки:", df.columns)
    df.printSchema()
    print("Строк:", df.count())

=== clients ===
Колонки: ['Client_ID', 'Покупатель', 'Contractor_name']
root
 |-- Client_ID: string (nullable = true)
 |-- Покупатель: string (nullable = true)
 |-- Contractor_name: string (nullable = true)

Строк: 3838
=== products ===
Колонки: ['Product_Key', 'Название товара', 'Название бренда', 'Группа товара']
root
 |-- Product_Key: string (nullable = true)
 |-- Название товара: string (nullable = true)
 |-- Название бренда: string (nullable = true)
 |-- Группа товара: string (nullable = true)

Строк: 21163
=== receipts ===
Колонки: ['Дата покупки', 'Client_ID', 'Адрес', 'Машина доставки', 'Филиал', 'Номер чека', 'Сумма покупки|Сумма']
root
 |-- Дата покупки: string (nullable = true)
 |-- Client_ID: string (nullable = true)
 |-- Адрес: string (nullable = true)
 |-- Машина доставки: string (nullable = true)
 |-- Филиал: string (nullable = true)
 |-- Номер чека: string (nullable = true)
 |-- Сумма покупки|Сумма: string (nullable = true)

Строк: 160114


In [ ]:
tx_clean.select("tx_date", "store", "client", "product", "sale_amount", "margin_pct").show(5, truncate=False)

big_sales = tx_clean.filter(F.col("sale_amount") >= 100)
print("Продаж >= 100:", big_sales.count())

tx_clean = tx_clean.withColumn(
    "discount_amount",
    F.round(F.col("amount_no_discount") - F.col("sale_amount"), 2)
)
print("Уникальных чеков:", tx_clean.select("receipt").distinct().count())
print("Уникальных товаров:", tx_clean.select("product").distinct().count())

+----------+-------+--------+---------+-----------+----------+
|tx_date   |store  |client  |product  |sale_amount|margin_pct|
+----------+-------+--------+---------+-----------+----------+
|2020-04-01|Store01|cl100636|sku006110|57.0       |30.7      |
|2020-04-01|Store01|cl100636|sku006114|49.0       |31.1      |
|2020-04-01|Store01|cl100636|sku006115|100.0      |18.52     |
|2020-04-01|Store01|cl100636|sku006130|38.0       |35.97     |
|2020-04-01|Store01|cl100636|sku006131|86.0       |45.81     |
+----------+-------+--------+---------+-----------+----------+
only showing top 5 rows
Продаж >= 100: 2106874
Уникальных чеков: 788616
Уникальных товаров: 49789


In [19]:
check_cols = ["tx_date", "client", "product", "sale_amount", "margin"]
null_counts = tx_clean.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in check_cols
])
null_counts.show(vertical=True)

-RECORD 0----------
 tx_date     | 0   
 client      | 0   
 product     | 0   
 sale_amount | 2   
 margin      | 0   



In [20]:
store_stats = (tx_clean
    .groupBy("store")
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("receipt").alias("receipts"),
        F.countDistinct("client").alias("clients"),
        F.round(F.sum("sale_amount"), 2).alias("revenue"),
        F.round(F.sum("margin"), 2).alias("total_margin"),
        F.round(F.avg("margin_pct"), 2).alias("avg_margin_pct")
    )
    .orderBy(F.desc("revenue")))
store_stats.show(truncate=False)

+-------+-------+--------+-------+------------+--------------+--------------+
|store  |rows   |receipts|clients|revenue     |total_margin  |avg_margin_pct|
+-------+-------+--------+-------+------------+--------------+--------------+
|Store01|2857011|672635  |131292 |1.77027948E9|3.1737617928E8|21.59         |
|Store02|419737 |115983  |36108  |2.7158701E8 |4.653712016E7 |10.25         |
+-------+-------+--------+-------+------------+--------------+--------------+



In [21]:
group_stats = (tx_clean
    .groupBy("group_name")
    .agg(
        F.count("*").alias("rows"),
        F.round(F.sum("sale_amount"), 2).alias("revenue"),
        F.round(F.avg("margin_pct"), 2).alias("avg_margin_pct"),
        F.sum("qty").alias("total_qty")
    )
    .orderBy(F.desc("revenue")))
group_stats.show(20, truncate=False)

+----------+------+------------+--------------+---------+
|group_name|rows  |revenue     |avg_margin_pct|total_qty|
+----------+------+------------+--------------+---------+
|group005  |222278|1.92530896E8|11.64         |984429   |
|group020  |117021|1.7373374E8 |9.51          |157065   |
|group009  |361800|1.6743914E8 |24.14         |595695   |
|group001  |75707 |1.4040684E8 |7.93          |81835    |
|group032  |117220|1.32594769E8|36.0          |524110   |
|group021  |343160|1.29196235E8|26.01         |485455   |
|group029  |33168 |1.12849458E8|16.0          |261882   |
|group010  |25753 |9.4403424E7 |5.89          |36280    |
|group019  |53255 |7.8133145E7 |15.95         |124464   |
|group006  |235508|7.476307E7  |27.86         |521765   |
|group024  |140909|7.3601998E7 |21.98         |252062   |
|group031  |23108 |6.9730435E7 |-2.74         |68982    |
|group012  |227570|5.8090305E7 |39.97         |471437   |
|group004  |29193 |5.6769887E7 |17.16         |95067    |
|group002  |49

In [ ]:
tx_keyed = (tx_clean
    .withColumn("client_key",  F.upper(F.regexp_replace("client", "_", "")))
    .withColumn("product_key", F.upper(F.regexp_replace("product", "_", ""))))

clients_key = (clients
    .withColumnRenamed("Client_ID", "client_id")
    .withColumn("client_key", F.upper(F.regexp_replace("client_id", "_", "")))
    .select("client_key", "client_id", "Покупатель", "Contractor_name"))

products_key = (products
    .withColumnRenamed("Product_Key", "product_id")
    .withColumn("product_key", F.upper(F.regexp_replace("product_id", "_", "")))
    .select("product_key", "product_id", "Название товара",
            "Название бренда", "Группа товара"))


enriched = (tx_keyed.alias("t")
    .join(clients_key.alias("c"), on="client_key", how="left")
    .join(products_key.alias("p"), on="product_key", how="left"))

print("Строк до JOIN:", tx_clean.count())
print("Строк после LEFT JOIN:", enriched.count())


inner_clients = tx_keyed.join(clients_key, on="client_key", how="inner")
print("Строк после INNER JOIN (clients):", inner_clients.count())


matched = enriched.filter(F.col("client_id").isNotNull()).count()
print("Сматчилось клиентов:", matched, "из", enriched.count())

Строк до JOIN: 3276748
Строк после LEFT JOIN: 3276748
Строк после INNER JOIN (clients): 0
Сматчилось клиентов: 0 из 3276748


In [24]:
enriched.createOrReplaceTempView("sales_enriched")

sql1 = spark.sql("""
    SELECT group_name,
           COUNT(*) AS items,
           COUNT(DISTINCT receipt) AS receipts,
           ROUND(SUM(sale_amount), 2) AS revenue,
           ROUND(AVG(margin_pct), 2) AS avg_margin_pct
    FROM sales_enriched
    GROUP BY group_name
    ORDER BY revenue DESC
    LIMIT 10
""")
sql1.show(truncate=False)

sql2 = spark.sql("""
    SELECT store,
           COUNT(DISTINCT client_id) AS unique_clients,
           ROUND(SUM(sale_amount), 2) AS revenue
    FROM sales_enriched
    GROUP BY store
    ORDER BY revenue DESC
""")
sql2.show(truncate=False)

+----------+------+--------+------------+--------------+
|group_name|items |receipts|revenue     |avg_margin_pct|
+----------+------+--------+------------+--------------+
|group005  |222278|136765  |1.92530896E8|11.64         |
|group020  |117021|76240   |1.7373374E8 |9.51          |
|group009  |361800|156833  |1.6743914E8 |24.14         |
|group001  |75707 |49750   |1.4040684E8 |7.93          |
|group032  |117220|57580   |1.32594769E8|36.0          |
|group021  |343160|177315  |1.29196235E8|26.01         |
|group029  |33168 |19469   |1.12849458E8|16.0          |
|group010  |25753 |19452   |9.4403424E7 |5.89          |
|group019  |53255 |42613   |7.8133145E7 |15.95         |
|group006  |235508|119062  |7.476307E7  |27.86         |
+----------+------+--------+------------+--------------+

+-------+--------------+------------+
|store  |unique_clients|revenue     |
+-------+--------------+------------+
|Store01|0             |1.77027948E9|
|Store02|0             |2.7158701E8 |
+-------+--

In [25]:
group_df = (enriched
    .groupBy("group_name")
    .agg(
        F.count("*").alias("items"),
        F.countDistinct("receipt").alias("receipts"),
        F.round(F.sum("sale_amount"), 2).alias("revenue"),
        F.round(F.avg("margin_pct"), 2).alias("avg_margin_pct")
    )
    .orderBy(F.desc("revenue")))
group_df.show(10, truncate=False)
print("Групп в витрине:", group_df.count())

+----------+------+--------+------------+--------------+
|group_name|items |receipts|revenue     |avg_margin_pct|
+----------+------+--------+------------+--------------+
|group005  |222278|136765  |1.92530896E8|11.64         |
|group020  |117021|76240   |1.7373374E8 |9.51          |
|group009  |361800|156833  |1.6743914E8 |24.14         |
|group001  |75707 |49750   |1.4040684E8 |7.93          |
|group032  |117220|57580   |1.32594769E8|36.0          |
|group021  |343160|177315  |1.29196235E8|26.01         |
|group029  |33168 |19469   |1.12849458E8|16.0          |
|group010  |25753 |19452   |9.4403424E7 |5.89          |
|group019  |53255 |42613   |7.8133145E7 |15.95         |
|group006  |235508|119062  |7.476307E7  |27.86         |
+----------+------+--------+------------+--------------+
only showing top 10 rows
Групп в витрине: 35


In [ ]:
OUT = BASE / "spark_output"
OUT.mkdir(exist_ok=True)


pdf = group_df.toPandas()
pdf.to_parquet(OUT / "group_sales.parquet", index=False)
pdf.to_csv(OUT / "group_sales.csv", index=False, encoding="utf-8")

print("Строк:", len(pdf))
print("Файлы записаны в", OUT.resolve())

Строк: 35
Файлы записаны в D:\Documents\Learning\7\ТИАБД\задания\3\spark_output


In [29]:
spark.stop()